In [2]:
# !pip install -r requirements.txt

In [7]:
import sys

repo_root = "/serafin/pcelayes/repos/sna_classifier"

sys.path.append(repo_root) # go to parent dir


In [9]:
from tw_dataset.settings import PROJECT_PATH, GT_GRAPH_PATH, NX_GRAPH_PATH, IG_GRAPH_PATH, DATASETS_FOLDER

In [11]:
import networkx as nx

In [12]:
graph = nx.read_graphml(IG_GRAPH_PATH)

In [15]:
len(graph.nodes())

5589

In [16]:
len(graph.edges())

261005

In [17]:
graph

In [ ]:
import torch
import networkx as nx
import numpy as np
from torch_geometric.utils import from_networkx, degree

def prepare_graph(G: nx.DiGraph):
    """
    Convert a NetworkX DiGraph to PyG format with structural node features.
    """
    # --- 1. Convert to PyG ---
    data = from_networkx(G)
    edge_index = data.edge_index  # [2, 260k]
    N = G.number_of_nodes()

    # --- 2. Build structural node features ---
    in_deg  = degree(edge_index[1], num_nodes=N)   # in-degree
    out_deg = degree(edge_index[0], num_nodes=N)   # out-degree
    total_deg = in_deg + out_deg

    # reciprocity per node: fraction of neighbors with mutual edges
    reciprocal_edges = sum(1 for u, v in G.edges() if G.has_edge(v, u))
    recip = torch.tensor(
        [sum(1 for nb in G.successors(n) if G.has_edge(nb, n)) / max(G.out_degree(n), 1)
         for n in G.nodes()],
        dtype=torch.float
    )

    # in/out degree ratio (direction bias per node)
    ratio = in_deg / (total_deg + 1e-8)

    # local clustering coefficient (undirected view)
    clustering = torch.tensor(
        [nx.clustering(G.to_undirected(), n) for n in G.nodes()],
        dtype=torch.float
    )

    # --- 3. Stack and normalize ---
    x = torch.stack([
        in_deg,
        out_deg,
        total_deg,
        ratio,        # 0 = source-like, 1 = sink-like
        recip,        # 0 = no reciprocal edges, 1 = all reciprocal
        clustering,
    ], dim=-1)  # [N, 6]

    # z-score normalize each feature
    mean = x.mean(dim=0, keepdim=True)
    std  = x.std(dim=0, keepdim=True) + 1e-8
    x = (x - mean) / std

    return edge_index, x, N

In [ ]:
import torch
import torch.nn.functional as F
from torch_geometric_signed_directed.nn.directed import MagNetConv
from torch_geometric.utils import negative_sampling

class MagNetUnsupervised(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, q=0.25, K=1):
        super().__init__()
        self.conv1 = MagNetConv(in_channels, hidden_channels, q=q, K=K)
        self.conv2 = MagNetConv(hidden_channels, hidden_channels, q=q, K=K)

    def encode(self, x, edge_index):
        x_r, x_i = self.conv1(x, x, edge_index)
        x_r, x_i = F.relu(x_r), F.relu(x_i)
        x_r, x_i = self.conv2(x_r, x_i, edge_index)
        # concatenate real and imaginary parts
        return torch.cat([x_r, x_i], dim=-1)  # [N, 2*hidden]

    def decode(self, z, edge_index):
        # dot product between source and target node embeddings
        return (z[edge_index[0]] * z[edge_index[1]]).sum(dim=-1)

    def forward(self, x, edge_index):
        z = self.encode(x, edge_index)
        # positive edges
        pos_scores = self.decode(z, edge_index)
        # negative samples (random non-edges)
        neg_edge_index = negative_sampling(edge_index, num_nodes=x.size(0),
                                           num_neg_samples=edge_index.size(1))
        neg_scores = self.decode(z, neg_edge_index)
        return pos_scores, neg_scores

In [ ]:
edge_index, x, N = prepare_graph(graph)

In [ ]:

# Training
model = MagNetUnsupervised(in_channels=F, hidden_channels=64)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

for epoch in range(200):
    model.train()
    optimizer.zero_grad()
    pos_scores, neg_scores = model(x, edge_index)
    loss = F.binary_cross_entropy_with_logits(
        torch.cat([pos_scores, neg_scores]),
        torch.cat([torch.ones(pos_scores.size(0)),
                   torch.zeros(neg_scores.size(0))])
    )
    loss.backward()
    optimizer.step()

# Extract embeddings
model.eval()
with torch.no_grad():
    embeddings = model.encode(x, edge_index)  # [5600, 128]